In [ ]:

from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from langchain_core.messages import HumanMessage, BaseMessage
from langgraph.graph.message import add_messages

from langgraph.prebuilt import ToolNode, tools_condition
from langchain_community.tools import DuckDuckGoSearch
from langchain_core.tools import tool

import requests
import random


llm = ChatGroq(
    model="openai/gpt-oss-20b"
)

search_tool = DuckDuckGoSearch(region="us-en")


@tool
def calculator(
    first_num: float,
    second_num: float,
    operation: str
) -> dict:
    """It performs basic mathematical operations"""

    try:
        if operation == "add":
            result = first_num + second_num

        elif operation == "sub":
            result = first_num - second_num

        elif operation == "multiply":
            result = first_num * second_num

        elif operation == "div":
            if second_num == 0:
                return {"error": "Division by zero is not allowed"}

            result = first_num / second_num

        else:
            return {"error": f"Unsupported operation '{operation}'"}

        return {
            "first_num": first_num,
            "second_num": second_num,
            "operation": operation,
            "result": result
        }

    except Exception as e:
        return {"error": str(e)}

tools = [search_tool,calculator]

llm_with_tools = llm.bind_tools(tools)

class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage],add_messages]
def chat_node (state:ChatState):
    """LLM that may answer or refer to the tool"""
    messages  = state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages":[response]}
     
tool_node = ToolNode(tool)


graph = StateGraph(ChatState)
graph.add_node("tools",tools)
graph.add_node("chat_node",chat_node)

graph.add_edge(START,"chat_node")
graph.add_conditional_edges("chat_node",tools_condition)
work_flow = graph.compile()
work_flow



